In [5]:
import nibabel as nib
import numpy as numpy
from scipy.ndimage import median_filter
from skimage import exposure
from skimage.restoration import denoise_bilateral
from scipy.ndimage import zoom
import vtk
from vtk.util import numpy_support


base = "/Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/B_FFE i0.9 XD Fast Gch/"
input_file = base + "1601_cropped_denoised.nii"
output_file = input_file[:-4] + ".vti"

mesh_offset = numpy.loadtxt(base + "Meshes/offset.txt")


In [6]:
def load_nifti(input_path, output_path, zoom_factor):
    """
    Crop a 3D NIfTI image and save it with updated affine.

    Parameters:
        input_path (str): input .nii/.nii.gz
        output_path (str): output file
        x_range, y_range, z_range: tuples (start, end)
    """

    nii = nib.load(input_path)
    data = numpy.array(nii.dataobj)  # efficient loading
    affine = nii.affine.copy()
    header = nii.header.copy()


    pixel_spacing = header.get_zooms()[:3] 
    spacing_file = input_path[:-4] + "_pixel_spacing.txt"
    
    with open(spacing_file, 'w') as f:
        # Writes it as space-separated values, or customize as you prefer
        f.write(f"{pixel_spacing[0]} {pixel_spacing[1]} {pixel_spacing[2]}\n")
    print(f"Original pixel spacing saved to: {spacing_file} -> {pixel_spacing}")


    print("Original matrix shape = ", data.shape)

    return data, pixel_spacing, affine[:,3]

data, pixel_spacing, affine = load_nifti(input_file, output_file, 1)
print("pixel_spacing = ", pixel_spacing)

Original pixel spacing saved to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/B_FFE i0.9 XD Fast Gch/1601_cropped_denoised_pixel_spacing.txt -> (np.float32(0.75000006), np.float32(0.75), np.float32(0.7500004))
Original matrix shape =  (90, 120, 80)
pixel_spacing =  (np.float32(0.75000006), np.float32(0.75), np.float32(0.7500004))


/var/folders/l0/d5k4byxx3690l9b12qr7mm440000gp/T/ipykernel_22848/414584402.py:12: DeprecationWarning: __array__ implementation doesn't accept a copy keyword, so passing copy=False failed. __array__ must implement 'dtype' and 'copy' keyword arguments. To learn more, see the migration guide https://numpy.org/devdocs/numpy_2_0_migration_guide.html#adapting-to-changes-in-the-copy-keyword
  data = numpy.array(nii.dataobj)  # efficient loading


In [7]:
def write_vti(data, pixel_spacing, offset, output_file):
    vtk_image = vtk.vtkImageData()
    vtk_image.SetDimensions(data.shape)
    vtk_image.SetSpacing(pixel_spacing)
    vtk_image.SetOrigin(offset[0], offset[1], offset[2])

    # Flatten in Fortran order to match VTK's x-fastest convention
    flat_data = data.flatten(order="F")
    vtk_array = numpy_support.numpy_to_vtk(flat_data, deep=True)
    vtk_array.SetName("intensity")
    vtk_image.GetPointData().SetScalars(vtk_array)

    writer = vtk.vtkXMLImageDataWriter()
    writer.SetFileName(output_file)
    writer.SetInputData(vtk_image)
    writer.Write()
    print(f"Saved VTI file to: {output_file}")

write_vti(data, pixel_spacing, -mesh_offset, output_file)

Saved VTI file to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/B_FFE i0.9 XD Fast Gch/1601_cropped_denoised.vti


## Segmentation mask

In [8]:
import nrrd

segment_folder = "segment/"
filename = "Segmentation.seg.nrrd"

eyeball_id = 3
background_id = -1


mask, header = nrrd.read(base + segment_folder + filename)
origin = numpy.array(header["space origin"])
print("origin = ", origin)

print(mask.shape)

write_vti(mask, pixel_spacing, -mesh_offset, base + segment_folder + filename[:-4] + ".vti" )

origin =  [ -63.27854919 -100.64540863   20.14351845]
(90, 120, 80)
Saved VTI file to: /Users/skardova/Documents/MRI_data/2026-Eyes-project/Trial2/B_FFE i0.9 XD Fast Gch/segment/Segmentation.seg..vti
